In [ ]:
# Step 1: Import Required Modules

import numpy as np
from numpy import multiply
from scipy.special import j0
from scipy.stats import poisson

from sequence.kernel.timeline import Timeline
from sequence.kernel.entity import Entity
from sequence.kernel.event import Event
from sequence.kernel.process import Process

from sequence.topology.node import QKDNode, Node

from sequence.components.optical_channel import QuantumChannel, ClassicalChannel
from sequence.components.light_source import LightSource
from sequence.components.photon import Photon, polarization
from sequence.components.detector import QSDetectorPolarization, Detector

from sequence.message import Message
from sequence.protocol import StackProtocol

from sequence.utils import log
from sequence.constants import SPEED_OF_LIGHT

import plotly.graph_objects as go

print("SPEED_OF_LIGHT:", SPEED_OF_LIGHT)
print("All imports OK")

In [ ]:
# ================= Step 2: Define Parameters =================
import numpy as np

MIMO_configs = [4, 8, 12]                     # editable
distances_km_graph = np.arange(2, 10 + 1, 2) # editable

mu_s, mu_1, mu_2 = 0.5, 0.1, 0.001    # decoy-state mean photon numbers (paper Sec. V)
decoy_probs = (0.5, 0.25, 0.25)       # selection probability for (mu_s, mu_1, mu_2)

p_m = 0.5    # Message Mode probability (confirmed)
p_c = 1 - p_m   # Control Mode probability

wavelength = 1550e-9
w = 0.035
ar = 0.20
delta = 0.43e-3
Cn2 = 1e-15
theta_p = 1e-6
eta_d = 0.12
Y0 = 1.6e-5
e_det = 0.015
q = 0.5
g_val = 1.03
e0 = 0.5

N_MIN_PHOTONS = 3   # paper: Bob generates n_i >= 3 photons (two-way feasibility, Sec. IV-A)
N_BITS = 20

print("MIMO_configs:", MIMO_configs)
print("distances_km_graph:", distances_km_graph)
print("mu_s, mu_1, mu_2:", mu_s, mu_1, mu_2, "| probs:", decoy_probs)
print("p_m:", p_m, "| p_c:", p_c)
print("Y0:", Y0, "| e_det:", e_det, "| eta_d:", eta_d)
print("N_MIN_PHOTONS (two-way feasibility):", N_MIN_PHOTONS)
print("N_BITS:", N_BITS)

In [ ]:
# ================= Step 3: Custom MIMO-FSO Quantum Channel Class (Two-Way, Round-Trip) =================
import numpy as np
from scipy.special import j0
from sequence.components.optical_channel import QuantumChannel

class MIMOFSOChannelTwoWay(QuantumChannel):
    """Quantum channel modeling the two-way MIMO-FSO round trip (Bob -> Alice -> Bob),
    following the paper's Eq. 21:
        T_i^(2-way) = eta_d * p_m * Ta_i * Tb_i * Tt_i^2 * beta_i^2
    where Ta_i, Tb_i are the atmospheric attenuation of each one-way leg
    (Bob->Alice and Alice->Bob respectively), Tt_i is the turbulence fading
    (experienced once per leg, hence squared), and beta_i is the MIMO SVD
    singular value (also experienced once per leg, hence squared).
    Also intercepts photons for Eve's PNS attack if an EveEntity is attached.
    """

    def __init__(self, name, timeline, distance, NT, NR,
                 wavelength=1550e-9, w=0.035, ar=0.20,
                 delta=0.43e-3, Cn2=1e-15, theta_p=1e-6, eta_d=0.12, p_m=0.5,
                 polarization_fidelity=1.0, light_speed=2e-4, frequency=8e7,
                 eve=None):

        super().__init__(name, timeline, attenuation=0, distance=distance,
                          polarization_fidelity=polarization_fidelity,
                          light_speed=light_speed, frequency=frequency)

        self.NT, self.NR = NT, NR
        self.wavelength, self.w, self.ar = wavelength, w, ar
        self.delta, self.Cn2, self.theta_p, self.eta_d = delta, Cn2, theta_p, eta_d
        self.p_m = p_m
        self.k = 2 * np.pi / wavelength

        self.betas = None
        self.Ti_all = None          # round-trip effective transmissivity (Eq. 21)
        self.eve = eve
        self.transmission_log = []

    def init(self) -> None:
        self.delay = round(self.distance / self.light_speed)
        z = self.distance
        aperture_area = np.pi * self.ar ** 2

        H, _, _ = self._build_H_matrix(z, aperture_area)
        self.betas = np.linalg.svd(H, compute_uv=False)

        Ta_i = 10 ** (-self.delta * z / 10)
        Tb_i = 10 ** (-self.delta * z / 10)

        d = self.ar * np.sqrt(self.k / z)
        chi2 = 1.23 * self.Cn2 * (self.k ** (7/6)) * (z ** (11/6))
        xi1 = 0.49*chi2 / (1 + 0.18*d**2 + 0.56*chi2**(12/5))**(7/6)
        xi2 = 0.51*chi2 / (1 + 0.9*d**2 + 0.62*(d**2)*chi2**(12/5))**(5/6)
        sigma2_turb = np.exp(xi1 + xi2) - 1
        Tt_i = np.exp(-0.5 * sigma2_turb)

        self.Ti_all = self.eta_d * self.p_m * Ta_i * Tb_i * (Tt_i**2) * (self.betas**2)
        self.loss = 1 - np.max(self.Ti_all)

        print(f"[{self.name}] init(): NT={self.NT} NR={self.NR} distance={z}m (round-trip leg)")
        print(f"[{self.name}] betas: {self.betas}")
        print(f"[{self.name}] Ta_i={Ta_i:.6f} Tb_i={Tb_i:.6f} Tt_i={Tt_i:.6f}")
        print(f"[{self.name}] Ti_all (2-way, Eq.21): {self.Ti_all}")
        print(f"[{self.name}] loss: {self.loss}")

    def calc_Tn(self, n, sub_channel_idx=None):
        Ti = np.max(self.Ti_all) if sub_channel_idx is None else self.Ti_all[sub_channel_idx]
        return 1 - (1 - Ti) ** n

    def transmit(self, qubit, source) -> None:
        if self.eve is not None:
            forward = self.eve.intercept(qubit)
            if not forward:
                print(f"[{self.name}] Photon '{qubit.name}' BLOCKED by Eve (stolen copy)")
                self.transmission_log.append((qubit.name, "blocked_by_eve"))
                return

        assert self.delay >= 0 and self.loss <= 1
        assert source == self.sender

        survives = (self.sender.get_generator().random() > self.loss) or qubit.is_null

        if survives:
            print(f"[{self.name}] Photon '{qubit.name}' TRANSMITTED (survived loss={self.loss:.4f}) in {self.delay} ps")
            self.transmission_log.append((qubit.name, "transmitted"))
            if qubit.is_null:
                qubit.add_loss(self.loss)
            if (qubit.encoding_type["name"] == "polarization"
                    and self.sender.get_generator().random() > self.polarization_fidelity):
                qubit.random_noise(self.get_generator())
            future_time = self.timeline.now() + self.delay
            process = Process(self.receiver, "receive_qubit", [source.name, qubit])
            event = Event(future_time, process)
            self.timeline.schedule(event)
        else:
            print(f"[{self.name}] Photon '{qubit.name}' LOST in channel (loss={self.loss:.4f})")
            self.transmission_log.append((qubit.name, "lost_in_channel"))

    def _build_H_matrix(self, z, aperture_area, n_points=60):
        R0 = np.sqrt(self.NT) * self.w
        tx_positions = self._square_grid_positions(self.NT, R0)
        rx_positions = self._square_grid_positions(self.NR, R0)
        r_points = np.linspace(0, R0, n_points)
        E_values = np.sqrt(2/(np.pi*self.w**2)) * np.exp(-r_points**2/self.w**2)
        rho_max = np.sin(self.wavelength/(np.pi*self.w))/self.wavelength
        rho_points = np.linspace(0, rho_max, n_points)
        F_values = np.zeros(n_points)
        for k_idx, rho in enumerate(rho_points):
            integrand = r_points * E_values * j0(2*np.pi*r_points*rho)
            F_values[k_idx] = 2*np.pi*np.trapezoid(integrand, r_points)
        norm_val = np.trapezoid(r_points * E_values**2, r_points)
        denom = np.sqrt(2*np.pi*norm_val)
        H = np.zeros((self.NR, self.NT), dtype=complex)
        for i in range(self.NR):
            for j in range(self.NT):
                dist = np.linalg.norm(rx_positions[i] - tx_positions[j])
                G_val = self._G_distance(dist, rho_points, F_values, z)
                H[i, j] = (aperture_area * G_val) / denom
        return H, tx_positions, rx_positions

    def _G_distance(self, r, rho_points, F_values, z):
        arg = self.k**2 - (2*np.pi*rho_points)**2
        phase = np.sqrt(np.maximum(arg, 0.0)) * z
        bessel = j0(2*np.pi*r*rho_points)
        real_integrand = rho_points * F_values * bessel * np.cos(phase)
        imag_integrand = rho_points * F_values * bessel * np.sin(phase)
        real_part = 2*np.pi*np.trapezoid(real_integrand, rho_points)
        imag_part = 2*np.pi*np.trapezoid(imag_integrand, rho_points)
        return real_part + 1j*imag_part

    @staticmethod
    def _square_grid_positions(N, R0):
        side = int(np.ceil(np.sqrt(N)))
        spacing = (2*R0)/(side+1)
        positions = []
        for ix in range(side):
            for iy in range(side):
                x = -R0 + spacing*(ix+1)
                y = -R0 + spacing*(iy+1)
                positions.append((x, y))
        return np.array(positions[:N])

In [ ]:
# ================= Step 4: Custom Decoy-State WCP Light Source (now on Bob's side) =================
from sequence.components.light_source import LightSource
from sequence.components.photon import Photon, polarization
from sequence.kernel.event import Event
from sequence.kernel.process import Process
from numpy import multiply

class DecoyWCPSource(LightSource):
    """Weak Coherent Pulse source with two-decoy-state method (Eq. 11).
    In the two-way (LM05) protocol, this source is attached to BOB (he
    initiates the exchange), not Alice. For each pulse, randomly selects
    mu_i from {mu_s, mu_1, mu_2}, samples photon number ~ Poisson(mu_i),
    and tags each emitted photon with its pulse index and total photon count.
    """

    def __init__(self, name, timeline, mu_s=0.5, mu_1=0.1, mu_2=0.001,
                 probs=(0.5, 0.25, 0.25), frequency=8e7, wavelength=1550,
                 bandwidth=0, encoding_type=None, phase_error=0):

        if encoding_type is None:
            encoding_type = polarization

        super().__init__(name, timeline, frequency=frequency, wavelength=wavelength,
                          bandwidth=bandwidth, mean_photon_num=mu_s,
                          encoding_type=encoding_type, phase_error=phase_error)

        self.mu_s, self.mu_1, self.mu_2 = mu_s, mu_1, mu_2
        self.probs = probs
        self.mu_sequence = []
        self.photon_count_log = []
        self.full_log = []

        self._state_to_bit_basis = {}
        for basis_idx in range(2):
            for bit_idx in range(2):
                state = self.encoding_type["bases"][basis_idx][bit_idx]
                self._state_to_bit_basis[state] = (basis_idx, bit_idx)

    def emit(self, state_list) -> None:
        print(f"[{self.name}] emit(): generating {len(state_list)} decoy-state WCP pulses")

        time = self.timeline.now()
        period = int(round(1e12 / self.frequency))
        mu_states = [self.mu_s, self.mu_1, self.mu_2]

        for i, state in enumerate(state_list):
            mu_i = self.get_generator().choice(mu_states, p=self.probs)
            self.mu_sequence.append(mu_i)

            num_photons = self.get_generator().poisson(mu_i)
            self.photon_count_log.append((i, mu_i, num_photons))

            basis_idx, bit_idx = self._state_to_bit_basis.get(
                tuple(state) if isinstance(state, list) else state, (None, None))
            self.full_log.append((i, bit_idx, basis_idx, mu_i, num_photons))
            print(f"  Pulse {i}: bit={bit_idx} basis={basis_idx} mu={mu_i:.3f} -> n_photons={num_photons}")

            if self.get_generator().random() < self.phase_error:
                state = multiply([1, -1], state)

            for copy_idx in range(num_photons):
                wl = self.linewidth * self.get_generator().standard_normal() + self.wavelength
                new_photon = Photon(str(i), self.timeline, wavelength=wl,
                                     location=self.owner, encoding_type=self.encoding_type,
                                     quantum_state=state)
                new_photon.pulse_index = i
                new_photon.n_total = num_photons
                new_photon.copy_index = copy_idx

                process = Process(self._receivers[0], "get", [new_photon])
                event = Event(time, process)
                self.timeline.schedule(event)
                self.photon_counter += 1

            time += period

In [ ]:
# ================= Step 5: Custom Eve Entity (PNS Attack, Both Phases of Round Trip) =================
class EveEntityTwoWay:
    """Models Eve performing a PNS attack in BOTH phases of the two-way LM05
    exchange (paper Sec. IV-A): Phase 1 (Bob->Alice, outbound) and Phase 2
    (Alice->Bob, return). Since the paper requires Bob to generate n_i >= 3
    photons for feasibility, Eve steals 1 copy in each phase (2 total),
    leaving at least 1 photon for Bob's final detection.
    """

    def __init__(self, name):
        self.name = name
        self.decided_pulses_phase1 = {}
        self.decided_pulses_phase2 = {}
        self.stored_photons_phase1 = []
        self.stored_photons_phase2 = []
        self.attack_log = []

    def intercept_phase1(self, photon):
        """Phase 1: Bob -> Alice (outbound). Steal 1 copy if n_total >= 3, forward the rest."""
        pulse_idx = getattr(photon, "pulse_index", None)
        n_total = getattr(photon, "n_total", 1)
        copy_idx = getattr(photon, "copy_index", 0)

        if pulse_idx is None:
            return True

        if n_total < 3:
            action = "forward (n<3, cannot safely attack both phases)"
            self.attack_log.append((pulse_idx, "phase1", n_total, copy_idx, action))
            print(f"[{self.name}] Phase1 Pulse {pulse_idx} (n={n_total}): {action}")
            return True

        if pulse_idx not in self.decided_pulses_phase1:
            self.decided_pulses_phase1[pulse_idx] = True
            self.stored_photons_phase1.append(pulse_idx)
            action = f"STOLEN (copy {copy_idx}) in Phase1 -> stored"
            self.attack_log.append((pulse_idx, "phase1", n_total, copy_idx, action))
            print(f"[{self.name}] Phase1 Pulse {pulse_idx} (n={n_total}): {action}")
            return False
        else:
            action = f"forwarded (copy {copy_idx}) toward Alice"
            self.attack_log.append((pulse_idx, "phase1", n_total, copy_idx, action))
            print(f"[{self.name}] Phase1 Pulse {pulse_idx} (n={n_total}): {action}")
            return True

    def intercept_phase2(self, photon):
        """Phase 2: Alice -> Bob (return, after Alice's MM/CM encoding).
        Steal 1 more copy if this pulse originally had n_total >= 3."""
        pulse_idx = getattr(photon, "pulse_index", None)
        n_total = getattr(photon, "n_total", 1)
        copy_idx = getattr(photon, "copy_index", 0)

        if pulse_idx is None:
            return True

        if n_total < 3:
            action = "forward (n<3, no second attack)"
            self.attack_log.append((pulse_idx, "phase2", n_total, copy_idx, action))
            print(f"[{self.name}] Phase2 Pulse {pulse_idx} (n={n_total}): {action}")
            return True

        if pulse_idx not in self.decided_pulses_phase2:
            self.decided_pulses_phase2[pulse_idx] = True
            self.stored_photons_phase2.append(pulse_idx)
            action = f"STOLEN (copy {copy_idx}) in Phase2 -> stored"
            self.attack_log.append((pulse_idx, "phase2", n_total, copy_idx, action))
            print(f"[{self.name}] Phase2 Pulse {pulse_idx} (n={n_total}): {action}")
            return False
        else:
            action = f"forwarded (copy {copy_idx}) toward Bob"
            self.attack_log.append((pulse_idx, "phase2", n_total, copy_idx, action))
            print(f"[{self.name}] Phase2 Pulse {pulse_idx} (n={n_total}): {action}")
            return True

In [ ]:
# ================= Step 6: Detector Parameters (Bob's threshold detector) =================
# Y0 and e_det already defined in Step 2. Built-in Detector supports dark_count
# natively; e_det will be modeled via the channel's polarization_fidelity,
# same approach as BB84.

print("Step 3-6 complete: MIMOFSOChannelTwoWay, DecoyWCPSource, EveEntityTwoWay defined.")
print(f"Detector: Y0={Y0}, e_det={e_det} (will configure built-in QSDetectorPolarization on Bob).")

In [ ]:
# ================= (Support class) One-way leg channel, used for both Phase 1 and Phase 2 =================
# Paste this as its own cell, right after Step 6, before Step 10.

class MIMOFSOChannelOneWayLeg(QuantumChannel):
    """One-way MIMO-FSO channel for a single leg of the LM05 round trip.
    Used twice: once for Bob->Alice (Phase 1), once for Alice->Bob (Phase 2).
    Eve attaches via intercept_fn (either eve.intercept_phase1 or eve.intercept_phase2).
    """

    def __init__(self, name, timeline, distance, NT, NR,
                 wavelength=1550e-9, w=0.035, ar=0.20,
                 delta=0.43e-3, Cn2=1e-15, theta_p=1e-6, eta_d=0.12,
                 polarization_fidelity=1.0, light_speed=2e-4, frequency=8e7,
                 intercept_fn=None):

        super().__init__(name, timeline, attenuation=0, distance=distance,
                          polarization_fidelity=polarization_fidelity,
                          light_speed=light_speed, frequency=frequency)

        self.NT, self.NR = NT, NR
        self.wavelength, self.w, self.ar = wavelength, w, ar
        self.delta, self.Cn2, self.theta_p, self.eta_d = delta, Cn2, theta_p, eta_d
        self.k = 2 * np.pi / wavelength

        self.betas = None
        self.Ti_all = None
        self.intercept_fn = intercept_fn
        self.transmission_log = []

    def init(self) -> None:
        self.delay = round(self.distance / self.light_speed)
        z = self.distance
        aperture_area = np.pi * self.ar ** 2

        H, _, _ = self._build_H_matrix(z, aperture_area)
        self.betas = np.linalg.svd(H, compute_uv=False)

        Ta = 10 ** (-self.delta * z / 10)

        d = self.ar * np.sqrt(self.k / z)
        chi2 = 1.23 * self.Cn2 * (self.k ** (7/6)) * (z ** (11/6))
        xi1 = 0.49*chi2 / (1 + 0.18*d**2 + 0.56*chi2**(12/5))**(7/6)
        xi2 = 0.51*chi2 / (1 + 0.9*d**2 + 0.62*(d**2)*chi2**(12/5))**(5/6)
        sigma2_turb = np.exp(xi1 + xi2) - 1
        Tt = np.exp(-0.5 * sigma2_turb)

        self.Ti_all = self.eta_d * Ta * Tt * self.betas
        self.loss = 1 - np.max(self.Ti_all)

        print(f"[{self.name}] init(): NT={self.NT} NR={self.NR} distance={z}m (one leg)")
        print(f"[{self.name}] Ti_all (one leg): {self.Ti_all}")
        print(f"[{self.name}] loss: {self.loss}")

    def transmit(self, qubit: "Photon", source) -> None:
        if self.intercept_fn is not None:
            forward = self.intercept_fn(qubit)
            if not forward:
                print(f"[{self.name}] Photon '{qubit.name}' BLOCKED by Eve")
                self.transmission_log.append((qubit.name, "blocked_by_eve"))
                return

        assert self.delay >= 0 and self.loss <= 1
        assert source == self.sender

        survives = (self.sender.get_generator().random() > self.loss) or qubit.is_null

        if survives:
            print(f"[{self.name}] Photon '{qubit.name}' TRANSMITTED (loss={self.loss:.4f}) in {self.delay} ps")
            self.transmission_log.append((qubit.name, "transmitted"))
            if qubit.is_null:
                qubit.add_loss(self.loss)
            if (qubit.encoding_type["name"] == "polarization"
                    and self.sender.get_generator().random() > self.polarization_fidelity):
                qubit.random_noise(self.get_generator())
            future_time = self.timeline.now() + self.delay
            process = Process(self.receiver, "receive_qubit", [source.name, qubit])
            event = Event(future_time, process)
            self.timeline.schedule(event)
        else:
            print(f"[{self.name}] Photon '{qubit.name}' LOST in channel (loss={self.loss:.4f})")
            self.transmission_log.append((qubit.name, "lost_in_channel"))

    def _build_H_matrix(self, z, aperture_area, n_points=60):
        R0 = np.sqrt(self.NT) * self.w
        tx_positions = self._square_grid_positions(self.NT, R0)
        rx_positions = self._square_grid_positions(self.NR, R0)
        r_points = np.linspace(0, R0, n_points)
        E_values = np.sqrt(2/(np.pi*self.w**2)) * np.exp(-r_points**2/self.w**2)
        rho_max = np.sin(self.wavelength/(np.pi*self.w))/self.wavelength
        rho_points = np.linspace(0, rho_max, n_points)
        F_values = np.zeros(n_points)
        for k_idx, rho in enumerate(rho_points):
            integrand = r_points * E_values * j0(2*np.pi*r_points*rho)
            F_values[k_idx] = 2*np.pi*np.trapezoid(integrand, r_points)
        norm_val = np.trapezoid(r_points * E_values**2, r_points)
        denom = np.sqrt(2*np.pi*norm_val)
        H = np.zeros((self.NR, self.NT), dtype=complex)
        for i in range(self.NR):
            for j in range(self.NT):
                dist = np.linalg.norm(rx_positions[i] - tx_positions[j])
                G_val = self._G_distance(dist, rho_points, F_values, z)
                H[i, j] = (aperture_area * G_val) / denom
        return H, tx_positions, rx_positions

    def _G_distance(self, r, rho_points, F_values, z):
        arg = self.k**2 - (2*np.pi*rho_points)**2
        phase = np.sqrt(np.maximum(arg, 0.0)) * z
        bessel = j0(2*np.pi*r*rho_points)
        real_integrand = rho_points * F_values * bessel * np.cos(phase)
        imag_integrand = rho_points * F_values * bessel * np.sin(phase)
        real_part = 2*np.pi*np.trapezoid(real_integrand, rho_points)
        imag_part = 2*np.pi*np.trapezoid(imag_integrand, rho_points)
        return real_part + 1j*imag_part

    @staticmethod
    def _square_grid_positions(N, R0):
        side = int(np.ceil(np.sqrt(N)))
        spacing = (2*R0)/(side+1)
        positions = []
        for ix in range(side):
            for iy in range(side):
                x = -R0 + spacing*(ix+1)
                y = -R0 + spacing*(iy+1)
                positions.append((x, y))
        return np.array(positions[:N])

In [ ]:
# Step 7: Create Timeline
from sequence.kernel.timeline import Timeline

tl = Timeline(10e12)

print("Timeline:", tl, "| now():", tl.now())

In [ ]:
# Step 8: Create Alice Node
from sequence.topology.node import QKDNode

alice = QKDNode("Alice", tl, stack_size=0)

print("Alice node:", alice.name)

In [ ]:
# Step 9: Create Bob Node
bob = QKDNode("Bob", tl, stack_size=0)

print("Bob node:", bob.name)

In [ ]:
# ================= Step 10: Attach MIMO-FSO Quantum Channels (Bob<->Alice) with Eve Interception =================

NT = MIMO_configs[0]                  # will be overwritten by the loop later
distance_km = distances_km_graph[0]   # will be overwritten by the loop later
distance_m = distance_km * 1000

# Step 5 (instantiate): Eve Entity, attached to BOTH legs
eve = EveEntityTwoWay("Eve")

# Phase 1: Bob -> Alice (outbound)
qc_b2a = MIMOFSOChannelOneWayLeg("qc_b2a", tl, distance=distance_m, NT=NT, NR=NT,
                                  polarization_fidelity=1 - e_det, intercept_fn=eve.intercept_phase1)
qc_b2a.set_ends(bob, alice.name)

# Phase 2: Alice -> Bob (return)
qc_a2b = MIMOFSOChannelOneWayLeg("qc_a2b", tl, distance=distance_m, NT=NT, NR=NT,
                                  polarization_fidelity=1 - e_det, intercept_fn=eve.intercept_phase2)
qc_a2b.set_ends(alice, bob.name)

print("NT = NR =", NT, "| distance =", distance_km, "km")
print("qc_b2a (Phase 1, Bob->Alice):", qc_b2a.name, "| Eve attached:", qc_b2a.intercept_fn is not None)
print("qc_a2b (Phase 2, Alice->Bob):", qc_a2b.name, "| Eve attached:", qc_a2b.intercept_fn is not None)

In [ ]:
# ================= Step 11: Attach Classical Channels =================
from sequence.components.optical_channel import ClassicalChannel

cc_a2b = ClassicalChannel("cc_a2b", tl, distance=distance_m)
cc_a2b.set_ends(alice, bob.name)

cc_b2a = ClassicalChannel("cc_b2a", tl, distance=distance_m)
cc_b2a.set_ends(bob, alice.name)

print("cc_a2b:", cc_a2b.name, "| distance:", cc_a2b.distance)
print("cc_b2a:", cc_b2a.name, "| distance:", cc_b2a.distance)

In [ ]:
# ================= Step 12: Attach Light Source to Bob (WCP originates from Bob) =================

bob_source = DecoyWCPSource("Bob.wcp_source", tl, mu_s=mu_s, mu_1=mu_1, mu_2=mu_2, probs=decoy_probs)
bob.add_component(bob_source)
bob_source.add_receiver(bob)   # emitted photons route through bob -> qchannel (qc_b2a)
bob.destination = "Alice"       # tells QKDNode.get() where to send photons (Phase 1)

print("Bob components:", list(bob.components.keys()))
print("Bob destination:", bob.destination)
print("Light source mu_s, mu_1, mu_2:", bob_source.mu_s, bob_source.mu_1, bob_source.mu_2)

In [ ]:
# ================= Step 13: Attach Alice's Encoding Logic (Message Mode / Control Mode) =================

class AliceEncoder:
    """Alice's LM05 encoding logic (paper Sec. IV-A). On receiving a qubit
    from Bob, Alice randomly chooses:
      - Message Mode (prob p_m): encode a random bit by applying Identity (bit=0)
        or a spin-flip / Pauli-Y operation (bit=1) to the qubit, WITHOUT measuring
        it, then sends the (possibly flipped) qubit back to Bob.
      - Control Mode (prob p_c): performs a projective measurement in a randomly
        chosen basis (Z or X) for eavesdropping detection; the qubit is consumed
        (not forwarded), so this pulse does not contribute to the key.
    """

    def __init__(self, name, owner, p_m, return_channel, rng):
        self.name = name
        self.owner = owner
        self.p_m = p_m
        self.return_channel = return_channel   # qc_a2b (Alice -> Bob)
        self.rng = rng
        self.mode_log = []    # (pulse_index, mode, bit_or_basis, outcome)

    def get(self, photon=None, **kwargs):
        if photon is None:
            return

        pulse_idx = getattr(photon, "pulse_index", None)
        is_mm = self.rng.random() < self.p_m

        if is_mm:
            bit = self.rng.integers(0, 2)   # 0 = Identity, 1 = spin-flip (Y)
            if bit == 1:
                a, b = photon.quantum_state.state if hasattr(photon.quantum_state, "state") else photon.quantum_state
                new_state = (-1j*b, 1j*a)
                photon.set_state(new_state)
                action = "Message Mode: bit=1 (spin-flip Y applied)"
            else:
                action = "Message Mode: bit=0 (Identity, unchanged)"

            self.mode_log.append((pulse_idx, "MM", bit, action))
            print(f"[{self.name}] Pulse {pulse_idx}: {action} -> forwarding to Bob")

            self.return_channel.transmit(photon, self.owner)

        else:
            basis_choice = self.rng.integers(0, 2)   # 0 = Z, 1 = X
            basis = photon.encoding_type["bases"][basis_choice]
            result = Photon.measure(basis, photon, self.rng)
            action = f"Control Mode: measured in basis={basis_choice} -> result={result}"
            self.mode_log.append((pulse_idx, "CM", basis_choice, action))   # was: result, now: action (fixed)
            print(f"[{self.name}] Pulse {pulse_idx}: {action} (qubit consumed, not forwarded)")

alice_encoder = AliceEncoder("Alice.encoder", alice, p_m, qc_a2b, alice.get_generator())
alice.add_component(alice_encoder)
alice.set_first_component(alice_encoder.name)

print("Alice's encoder attached:", alice_encoder.name)
print("Alice first_component_name:", alice.first_component_name)

In [ ]:
# ================= Step 14: Attach Threshold Detector to Bob (final detection, returned photon) =================
from sequence.components.detector import QSDetectorPolarization

bob_qsdetector = QSDetectorPolarization("Bob.qsdetector_custom", tl)
for d in bob_qsdetector.detectors:
    d.dark_count = Y0   # background/dark count rate (paper Sec. V)

bob.add_component(bob_qsdetector)
for d in bob_qsdetector.detectors:
    bob.add_component(d)
bob.add_component(bob_qsdetector.splitter)
bob.set_first_component(bob_qsdetector.name)   # routes incoming qubits (Phase 2 return) here

print("Bob components:", list(bob.components.keys()))
print("Bob first_component_name:", bob.first_component_name)
print("Detector 0 dark_count:", bob_qsdetector.detectors[0].dark_count)
print("Detector 1 dark_count:", bob_qsdetector.detectors[1].dark_count)

In [ ]:
# ================= Step 14 (patch): Bob's Detector Click/No-Click Logging =================
# Insert this right after Step 14 (Attach Threshold Detector to Bob), before Step 15.

detection_log = []   # (detector_label, photon_name, outcome)

def make_logging_get(detector, label):
    original_record = detector.record_detection
    def logging_get(photon=None, **kwargs):
        detector.photon_counter += 1
        if photon and photon.encoding_type["name"] == "single_atom":
            key = photon.quantum_state
            res = detector.timeline.quantum_manager.run_circuit(
                type(detector)._meas_circuit, [key], detector.get_generator().random())
            if not res[key]:
                detection_log.append((label, photon.name if photon else "?", "NO-CLICK (measured |0>)"))
                return
        if detector.get_generator().random() < detector.efficiency:
            detection_log.append((label, photon.name if photon else "?", "CLICK"))
            original_record()
        else:
            detection_log.append((label, photon.name if photon else "?", "NO-CLICK (detector inefficiency)"))
    return logging_get

bob_qsdetector.detectors[0].get = make_logging_get(bob_qsdetector.detectors[0], "Bob.detector0")
bob_qsdetector.detectors[1].get = make_logging_get(bob_qsdetector.detectors[1], "Bob.detector1")

print("Bob's detectors patched for click/no-click logging (stored in detection_log).")

In [ ]:
# ================= Step 15: Create LM05 Protocol Instances =================
# No built-in sequence.qkd.LM05 exists, so this is a custom orchestrator that
# ties together Bob's source, Alice's encoder, Bob's detector, and Eve --
# manages pulse generation, round-trip execution, and final key extraction.

class LM05Protocol:
    def __init__(self, name, alice, bob, bob_source, alice_encoder, bob_qsdetector, eve, N_BITS):
        self.name = name
        self.alice = alice
        self.bob = bob
        self.bob_source = bob_source
        self.alice_encoder = alice_encoder
        self.bob_qsdetector = bob_qsdetector
        self.eve = eve
        self.N_BITS = N_BITS

        self.bob_key_bits = []      # Bob's inferred bits (from MM-mode pulses only)
        self.alice_key_bits = []    # Alice's originally encoded bits (ground truth)

lm05 = LM05Protocol("lm05", alice, bob, bob_source, alice_encoder, bob_qsdetector, eve, N_BITS)

print("LM05 protocol orchestrator created:", lm05.name)
print("Linked: bob_source =", lm05.bob_source.name, "| alice_encoder =", lm05.alice_encoder.name)

In [ ]:
# ================= Step 16: Pair LM05 Protocols =================
# Since LM05Protocol is a single orchestrator (not two separate sender/receiver
# instances like BB84), "pairing" here means explicitly assigning roles and
# registering the orchestrator on both nodes.

lm05.role_bob = "initiator"    # Bob generates WCP, sends first (Phase 1)
lm05.role_alice = "responder"  # Alice encodes and returns (Phase 2)

alice.protocols.append(lm05)
bob.protocols.append(lm05)

print("LM05 paired:")
print("  Bob role:", lm05.role_bob)
print("  Alice role:", lm05.role_alice)
print("  Alice protocols:", [p.name for p in alice.protocols])
print("  Bob protocols:", [p.name for p in bob.protocols])

In [ ]:
# Step 17: Initialize Timeline
tl.init()

print("Timeline initialized successfully")

In [ ]:
# ================= Step 18: Show Channel Parameters (Round-Trip Ti, Eq. 21) =================
# Instantiate the analytical two-way channel (Eq. 21) for comparison/QBER-SKR use.
# This is NOT attached to any node -- it's a standalone calculator using the
# same physical parameters as the two simulated one-way legs.

qc_twoway = MIMOFSOChannelTwoWay("qc_twoway_analytical", tl, distance=distance_m, NT=NT, NR=NT, p_m=p_m)
qc_twoway.init()

print(f"===== Channel Parameters (NT=NR={NT}, distance={distance_km} km) =====")
print("Phase 1 (Bob->Alice) Ti (one leg):", qc_b2a.Ti_all)
print("Phase 2 (Alice->Bob) Ti (one leg):", qc_a2b.Ti_all)
print("Round-trip Ti (Eq. 21, analytical):", qc_twoway.Ti_all)

In [ ]:
# ================= Step 19: Set Key Length & Push Protocol Request =================
# Per the paper (Sec. IV-A): "Bob encodes the classical information bit using
# the LM05 protocol and transmits the prepared qubit to Alice." This means
# bob_bit_list below IS Bob's real classical information bit for each pulse --
# not a placeholder. Bob remembers both his basis and his bit; after the round
# trip, he measures in the SAME basis and compares against this original bit
# to detect whether Alice applied Identity/spin-flip (and to check for Eve).

rng = bob.get_generator()

Ti_est = np.max(qc_twoway.Ti_all)
N_PULSES = int(np.ceil(N_BITS / max(Ti_est, 1e-6)) * 3)   # 3x safety margin for round-trip loss

bob_basis_list = rng.choice([0, 1], N_PULSES)   # basis Bob prepares each qubit in (Z=0, X=1)
bob_bit_list = rng.choice([0, 1], N_PULSES)     # Bob's own classical information bit (paper Sec. IV-A)

state_list = [polarization["bases"][bob_basis_list[i]][bob_bit_list[i]] for i in range(N_PULSES)]

print("N_BITS requested:", N_BITS)
print("Estimated round-trip Ti:", Ti_est)
print("N_PULSES to send (with safety margin):", N_PULSES)
print("Bob's basis list (first 20):", bob_basis_list[:20])
print("Bob's classical info bit list (first 20):", bob_bit_list[:20])

In [ ]:
# ================= Step 19 (patch): Configure Bob's Detector Basis List =================
# Bob must measure the RETURNED photon in the SAME basis he originally
# prepared it in (bob_basis_list from Step 19) -- this is how he detects
# whether Alice applied Identity or spin-flip. The splitter's basis_list is
# indexed by (arrival_time - start_time) * frequency, so start_time MUST
# account for the full round-trip delay (Bob->Alice->Bob), otherwise the
# computed index falls outside the list and the splitter silently drops
# every photon (this was the actual root cause of 0 clicks).

round_trip_delay_est = qc_b2a.delay + qc_a2b.delay   # approximate round-trip delay in ps

bob_qsdetector.set_basis_list(list(bob_basis_list), round_trip_delay_est, bob_source.frequency)

print("Bob's detector basis_list configured:", len(bob_qsdetector.splitter.basis_list), "entries")
print("round_trip_delay_est (start_time):", round_trip_delay_est, "ps")
print("frequency:", bob_qsdetector.splitter.frequency)

In [ ]:
# ================= Step 20: Run Timeline (Execute Full Protocol) =================
# We manually schedule Bob's emit() call at time 0, since there's no built-in
# LM05 protocol class to auto-trigger it (unlike BB84's begin_photon_pulse).
# Photon generation, transmission, Eve's PNS attack (both phases), Alice's
# MM/CM encoding, and Bob's final detection all fire automatically during
# tl.run(), via the print statements built into each class.

process = Process(bob_source, "emit", [state_list])
event = Event(0, process)
tl.schedule(event)

tl.run()

print()
print("Timeline run complete. Final simulation time:", tl.now())

In [ ]:
# ================= Step 21: Display Photon Generation Log (Bob's WCP) =================
# Uses bob_source.full_log (stored during Step 20's run) for a clean summary,
# instead of scrolling through the live output.

print("===== Bob's Photon Generation Log (WCP) =====")
print("Total pulses generated:", len(bob_source.full_log))
print()
print("Sample (first 10 pulses):")
for pulse_idx, bit, basis, mu, n_photons in bob_source.full_log[:10]:
    print(f"  Pulse {pulse_idx}: bit={bit} basis={basis} mu={mu:.3f} -> n_photons={n_photons}")

n_zero = sum(1 for _,_,_,_,n in bob_source.full_log if n == 0)
n_one = sum(1 for _,_,_,_,n in bob_source.full_log if n == 1)
n_multi = sum(1 for _,_,_,_,n in bob_source.full_log if n > 1)

print()
print("Vacuum (n=0):", n_zero, "| Single-photon (n=1):", n_one, "| Multi-photon (n>1):", n_multi)

In [ ]:
# ================= Step 22: Display Photon Transmission Log (Bob -> Alice, Phase 1) =================

print("===== Photon Transmission Log (Phase 1: Bob -> Alice) =====")
print("Total transmission attempts logged:", len(qc_b2a.transmission_log))
print()
print("Sample (first 10):")
for name, outcome in qc_b2a.transmission_log[:10]:
    print(f"  Photon '{name}': {outcome}")

blocked = sum(1 for _, o in qc_b2a.transmission_log if o == "blocked_by_eve")
transmitted = sum(1 for _, o in qc_b2a.transmission_log if o == "transmitted")
lost = sum(1 for _, o in qc_b2a.transmission_log if o == "lost_in_channel")

print()
print("Blocked by Eve (Phase 1):", blocked)
print("Transmitted (reached Alice):", transmitted)
print("Lost in channel:", lost)
print("Total:", blocked + transmitted + lost)

In [ ]:
# ================= Step 23: Display Alice's MM/CM Decision Log =================

print("===== Alice's Message Mode / Control Mode Decision Log =====")
print("Total decisions logged:", len(alice_encoder.mode_log))
print()
print("Sample (first 10):")
for pulse_idx, mode, bit_or_basis, action in alice_encoder.mode_log[:10]:
    print(f"  Pulse {pulse_idx}: {action}")

n_mm = sum(1 for _, mode, _, _ in alice_encoder.mode_log if mode == "MM")
n_cm = sum(1 for _, mode, _, _ in alice_encoder.mode_log if mode == "CM")

print()
print("Message Mode (MM) count:", n_mm, f"(~{100*n_mm/len(alice_encoder.mode_log):.1f}%)")
print("Control Mode (CM) count:", n_cm, f"(~{100*n_cm/len(alice_encoder.mode_log):.1f}%)")
print("(expected split: p_m =", p_m, "| p_c =", p_c, ")")

In [ ]:
# ================= Step 24: Display Eve's PNS Attack Log (Both Phases) =================

print("===== Eve's PNS Attack Log (Phase 1 + Phase 2) =====")
print("Total interceptions logged:", len(eve.attack_log))
print()
print("Sample (first 10):")
for pulse_idx, phase, n_total, copy_idx, action in eve.attack_log[:10]:
    print(f"  Pulse {pulse_idx} [{phase}] (n={n_total}): {action}")

phase1_entries = [e for e in eve.attack_log if e[1] == "phase1"]
phase2_entries = [e for e in eve.attack_log if e[1] == "phase2"]

print()
print("Phase 1 (Bob->Alice) interceptions:", len(phase1_entries))
print("  Stolen in Phase 1:", len(eve.stored_photons_phase1))
print("Phase 2 (Alice->Bob) interceptions:", len(phase2_entries))
print("  Stolen in Phase 2:", len(eve.stored_photons_phase2))

both_phases_stolen = set(eve.stored_photons_phase1) & set(eve.stored_photons_phase2)
print()
print("Pulses where Eve stole in BOTH phases (full attack):", len(both_phases_stolen))
print("  Pulse indices:", list(both_phases_stolen)[:10])

In [ ]:
# ================= Step 25: Display Photon Forwarding Log (Alice -> Bob, Phase 2 Return Trip) =================

print("===== Photon Forwarding Log (Phase 2: Alice -> Bob, Return Trip) =====")
print("Total transmission attempts logged:", len(qc_a2b.transmission_log))
print()
print("Sample (first 10):")
for name, outcome in qc_a2b.transmission_log[:10]:
    print(f"  Photon '{name}': {outcome}")

blocked = sum(1 for _, o in qc_a2b.transmission_log if o == "blocked_by_eve")
transmitted = sum(1 for _, o in qc_a2b.transmission_log if o == "transmitted")
lost = sum(1 for _, o in qc_a2b.transmission_log if o == "lost_in_channel")

print()
print("Blocked by Eve (Phase 2):", blocked)
print("Transmitted (reached Bob):", transmitted)
print("Lost in channel:", lost)
print("Total:", blocked + transmitted + lost)

In [ ]:
# ================= Step 26: Display Bob's Reception & Detection Log =================
print("===== Bob's Reception & Detection Log (Final, after round trip) =====")
print("Total detection attempts logged:", len(detection_log))
for label, photon_name, outcome in detection_log[:10]:
    print(f"  [{label}] Photon '{photon_name}': {outcome}")

clicks = sum(1 for _, _, o in detection_log if o == "CLICK")
no_clicks = len(detection_log) - clicks
print("Total CLICKs:", clicks, "| Total NO-CLICKs:", no_clicks)

In [ ]:
# ================= Step 27: Display Sifting Log =================
# Sifting in LM05: only Message Mode pulses that (a) survived the full round
# trip AND (b) registered a CLICK at Bob's detector contribute to the final
# key. Control Mode pulses never contribute (used only for eavesdropping
# detection). This correlates alice_encoder.mode_log with detection_log.

print("===== Sifting Log (LM05) =====")

# pulse indices that were Message Mode
mm_pulse_indices = set(p for p, mode, _, _ in alice_encoder.mode_log if mode == "MM")

# pulse indices that got a CLICK at Bob (photon name == str(pulse_index))
clicked_pulse_names = set(name for _, name, outcome in detection_log if outcome == "CLICK")
clicked_pulse_indices = set(int(n) for n in clicked_pulse_names if n.isdigit())

sifted_indices = mm_pulse_indices & clicked_pulse_indices

print("Message Mode pulses:", len(mm_pulse_indices))
print("Clicked pulses (any mode):", len(clicked_pulse_indices))
print("Sifted (MM AND clicked):", len(sifted_indices))
print("Sifted pulse indices:", sifted_indices)

In [ ]:
# ================= Step 28: Display Final Key (Bob vs Alice, Match Check) =================
# Bob infers Alice's encoded bit as: measured_bit XOR bob_original_bit
# (Identity -> measured==original -> XOR=0; spin-flip -> measured!=original -> XOR=1).
# This is compared against Alice's actual recorded bit (ground truth) for verification.

measured_bit_by_pulse = {}
for label, photon_name, outcome in detection_log:
    if outcome == "CLICK" and photon_name.isdigit():
        pulse_idx = int(photon_name)
        measured_bit = 0 if "detector0" in label else 1
        measured_bit_by_pulse[pulse_idx] = measured_bit

alice_bit_by_pulse = {p: bit for p, mode, bit, _ in alice_encoder.mode_log if mode == "MM"}

bob_key_bits = []
alice_key_bits = []

print("===== Final Key (LM05) =====")
for idx in sorted(sifted_indices):
    if idx in measured_bit_by_pulse:
        measured = measured_bit_by_pulse[idx]
        original = bob_bit_list[idx]
        bob_inferred_bit = measured ^ original
        alice_actual_bit = alice_bit_by_pulse.get(idx)

        bob_key_bits.append(bob_inferred_bit)
        alice_key_bits.append(alice_actual_bit)

        print(f"  Pulse {idx}: Bob_original={original} measured={measured} -> Bob_inferred_bit={bob_inferred_bit} | Alice_actual_bit={alice_actual_bit}")

print()
print("Bob's final key:  ", bob_key_bits)
print("Alice's final key:", alice_key_bits)
print("Key length:", len(bob_key_bits))

if len(bob_key_bits) > 0:
    matches = sum(1 for b, a in zip(bob_key_bits, alice_key_bits) if b == a)
    print("Matching bits:", matches, "/", len(bob_key_bits))
    print("Keys match exactly:", bob_key_bits == alice_key_bits)
else:
    print("No sifted key bits yet -- expected with current NT=1, small N_PULSES.")

In [ ]:
# ================= Step 29: Compute QBER (Paper Formula, Eq. 22-23, Two-Way) =================
# Paper's two-way QBER analog is E_tilde_{mu_s,i} (Eq. 23), using the
# round-trip transmissivity T_i^(2-way) (Eq. 21) instead of the one-way Ti.

def calc_Q_tilde(mu, Ti_2way, Y0):
    """Q_tilde_{mu_s,i} (Eq. 23, first line)."""
    return Y0 + (1 - Y0) * (1 - np.exp(-mu * Ti_2way))

def calc_E_tilde(mu, Ti_2way, Y0, e0, e_det):
    """E_tilde_{mu_s,i} (Eq. 23, second line) -- the two-way QBER analog."""
    Q_tilde = calc_Q_tilde(mu, Ti_2way, Y0)
    return (1 / Q_tilde) * (e0*Y0 + e_det*(1 - np.exp(-mu*Ti_2way)))

Ti_2way_all = qc_twoway.Ti_all

E_tilde_all = np.array([calc_E_tilde(mu_s, Ti, Y0, e0, e_det) for Ti in Ti_2way_all])
Q_tilde_all = np.array([calc_Q_tilde(mu_s, Ti, Y0) for Ti in Ti_2way_all])

QBER_2way_MIMO = np.sum(E_tilde_all * Q_tilde_all) / np.sum(Q_tilde_all)

print("Ti_2way_all (round-trip, per sub-channel):", Ti_2way_all)
print("Q_tilde_all:", Q_tilde_all)
print("E_tilde_all (two-way QBER, per sub-channel):", E_tilde_all)
print("QBER_2way_MIMO (weighted average):", QBER_2way_MIMO)

In [ ]:
# ================= Step 30: Compute SKR (Paper Formula, Eq. 22, Two-Way) =================
# NOTE: the paper's visible text does not give explicit Q_{n,i}^L / e_tilde_{n,i}
# sub-formulas for n=1,2 in the two-way case (unlike one-way's fully-specified
# Eq. 17-18). This extends the same Tn,i pattern (Eq. 12, stated generically)
# applied to T_i^(2-way), as the most consistent available approach.

def H2(x):
    x = np.clip(x, 1e-12, 1-1e-12)
    return -x*np.log2(x) - (1-x)*np.log2(1-x)

def G_func(e_tilde):
    if e_tilde < 0.5:
        val = 1 + 4*e_tilde - 4*e_tilde**2
        return np.log2(max(val, 1e-12))
    return 1.0

def calc_Tn_2way(Ti_2way, n):
    return 1 - (1 - Ti_2way) ** n

def calc_Qn_L(Ti_2way, n, mu_s, Y0):
    from scipy.stats import poisson as poisson_dist
    Tn = calc_Tn_2way(Ti_2way, n)
    Yn = Y0 + (1 - Y0) * Tn
    Pn = poisson_dist.pmf(n, mu_s)
    return Yn * Pn

def calc_e_tilde_n(Ti_2way, n, Y0, e0, e_det):
    Tn = calc_Tn_2way(Ti_2way, n)
    Yn = Y0 + (1 - Y0) * Tn
    return (e0*Y0 + e_det*Tn) / Yn

def calc_SKR_i_2way(Ti_2way, mu_s, Y0, e0, e_det, q, g_val):
    Q_tilde = calc_Q_tilde(mu_s, Ti_2way, Y0)
    E_tilde = calc_E_tilde(mu_s, Ti_2way, Y0, e0, e_det)
    term1 = -Q_tilde * g_val * H2(E_tilde)
    term2 = 0
    for n in [1, 2]:
        Qn_L = calc_Qn_L(Ti_2way, n, mu_s, Y0)
        e_tilde_n = calc_e_tilde_n(Ti_2way, n, Y0, e0, e_det)
        term2 += Qn_L * (1 - G_func(e_tilde_n))
    return q * (term1 + term2)

SKR_2way_i_all = np.array([calc_SKR_i_2way(Ti, mu_s, Y0, e0, e_det, q, g_val) for Ti in Ti_2way_all])
SKR_2way_MIMO = np.sum(SKR_2way_i_all)

print("SKR per sub-channel (2-way):", SKR_2way_i_all)
print("SKR_2way_MIMO (paper formula, Eq. 22):", SKR_2way_MIMO)

In [ ]:
# ================= Function combining Steps 7-30 (paste this BEFORE the Step 31 loop cell) =================
import numpy as np
from sequence.kernel.timeline import Timeline
from sequence.topology.node import QKDNode
from sequence.components.optical_channel import ClassicalChannel
from sequence.components.detector import QSDetectorPolarization
from sequence.components.photon import Photon, polarization
from sequence.kernel.event import Event
from sequence.kernel.process import Process


def run_lm05_mimo_fso_simulation(NT, distance_km, verbose=True):
    """Runs one full SeQUeNCe LM05 (two-way) + MIMO-FSO + PNS-attack (both
    phases) simulation for the given MIMO configuration and distance.
    Combines Steps 7-30. Returns a dict of results.
    """
    NR = NT
    distance_m = distance_km * 1000

    tl = Timeline(10e12)
    alice = QKDNode("Alice", tl, stack_size=0)
    bob = QKDNode("Bob", tl, stack_size=0)

    eve = EveEntityTwoWay("Eve")

    qc_b2a = MIMOFSOChannelOneWayLeg("qc_b2a", tl, distance=distance_m, NT=NT, NR=NR,
                                      polarization_fidelity=1 - e_det, intercept_fn=eve.intercept_phase1)
    qc_b2a.set_ends(bob, alice.name)
    qc_a2b = MIMOFSOChannelOneWayLeg("qc_a2b", tl, distance=distance_m, NT=NT, NR=NR,
                                      polarization_fidelity=1 - e_det, intercept_fn=eve.intercept_phase2)
    qc_a2b.set_ends(alice, bob.name)

    cc_a2b = ClassicalChannel("cc_a2b", tl, distance=distance_m)
    cc_a2b.set_ends(alice, bob.name)
    cc_b2a = ClassicalChannel("cc_b2a", tl, distance=distance_m)
    cc_b2a.set_ends(bob, alice.name)

    bob_source = DecoyWCPSource("Bob.wcp_source", tl, mu_s=mu_s, mu_1=mu_1, mu_2=mu_2, probs=decoy_probs)
    bob.add_component(bob_source)
    bob_source.add_receiver(bob)
    bob.destination = "Alice"

    alice_encoder = AliceEncoder("Alice.encoder", alice, p_m, qc_a2b, alice.get_generator())
    alice.add_component(alice_encoder)
    alice.set_first_component(alice_encoder.name)

    bob_qsdetector = QSDetectorPolarization("Bob.qsdetector_custom", tl)
    for d in bob_qsdetector.detectors:
        d.dark_count = Y0
    bob.add_component(bob_qsdetector)
    for d in bob_qsdetector.detectors:
        bob.add_component(d)
    bob.add_component(bob_qsdetector.splitter)
    bob.set_first_component(bob_qsdetector.name)

    detection_log = []
    def make_logging_get(detector, label):
        original_record = detector.record_detection
        def logging_get(photon=None, **kwargs):
            detector.photon_counter += 1
            if detector.get_generator().random() < detector.efficiency:
                detection_log.append((label, photon.name if photon else "?", "CLICK"))
                original_record()
            else:
                detection_log.append((label, photon.name if photon else "?", "NO-CLICK"))
        return logging_get
    bob_qsdetector.detectors[0].get = make_logging_get(bob_qsdetector.detectors[0], "Bob.detector0")
    bob_qsdetector.detectors[1].get = make_logging_get(bob_qsdetector.detectors[1], "Bob.detector1")

    lm05 = LM05Protocol("lm05", alice, bob, bob_source, alice_encoder, bob_qsdetector, eve, N_BITS)
    alice.protocols.append(lm05)
    bob.protocols.append(lm05)

    tl.init()

    qc_twoway = MIMOFSOChannelTwoWay("qc_twoway_analytical", tl, distance=distance_m, NT=NT, NR=NR, p_m=p_m)
    qc_twoway.init()

    if verbose:
        print(f"\n===== NT=NR={NT}, distance={distance_km} km =====")
        print("Round-trip Ti (Eq.21):", qc_twoway.Ti_all)

    rng = bob.get_generator()
    Ti_est = np.max(qc_twoway.Ti_all)
    N_PULSES = int(np.ceil(N_BITS / max(Ti_est, 1e-6)) * 3)

    bob_basis_list = rng.choice([0, 1], N_PULSES)
    bob_bit_list = rng.choice([0, 1], N_PULSES)
    state_list = [polarization["bases"][bob_basis_list[i]][bob_bit_list[i]] for i in range(N_PULSES)]

    round_trip_delay_est = qc_b2a.delay + qc_a2b.delay
    bob_qsdetector.set_basis_list(list(bob_basis_list), round_trip_delay_est, bob_source.frequency)

    process = Process(bob_source, "emit", [state_list])
    event = Event(0, process)
    tl.schedule(event)
    tl.run()

    mm_pulse_indices = set(p for p, mode, _, _ in alice_encoder.mode_log if mode == "MM")
    clicked_pulse_names = set(name for _, name, outcome in detection_log if outcome == "CLICK")
    clicked_pulse_indices = set(int(n) for n in clicked_pulse_names if n.isdigit())
    sifted_indices = mm_pulse_indices & clicked_pulse_indices

    measured_bit_by_pulse = {}
    for label, photon_name, outcome in detection_log:
        if outcome == "CLICK" and photon_name.isdigit():
            measured_bit_by_pulse[int(photon_name)] = 0 if "detector0" in label else 1
    alice_bit_by_pulse = {p: bit for p, mode, bit, _ in alice_encoder.mode_log if mode == "MM"}

    bob_key_bits, alice_key_bits = [], []
    for idx in sorted(sifted_indices):
        if idx in measured_bit_by_pulse:
            measured = measured_bit_by_pulse[idx]
            bob_inferred = measured ^ bob_bit_list[idx]
            bob_key_bits.append(bob_inferred)
            alice_key_bits.append(alice_bit_by_pulse.get(idx))

    key_match = (bob_key_bits == alice_key_bits) and len(bob_key_bits) > 0

    Ti_2way_all = qc_twoway.Ti_all
    Q_tilde_all = np.array([calc_Q_tilde(mu_s, Ti, Y0) for Ti in Ti_2way_all])
    E_tilde_all = np.array([calc_E_tilde(mu_s, Ti, Y0, e0, e_det) for Ti in Ti_2way_all])
    QBER_2way_MIMO = np.sum(E_tilde_all * Q_tilde_all) / np.sum(Q_tilde_all)

    SKR_2way_i_all = np.array([calc_SKR_i_2way(Ti, mu_s, Y0, e0, e_det, q, g_val) for Ti in Ti_2way_all])
    SKR_2way_MIMO = np.sum(SKR_2way_i_all)

    if verbose:
        print(f"Sifted key length: {len(bob_key_bits)} | key_match: {key_match}")
        print(f"QBER_2way_MIMO: {QBER_2way_MIMO} | SKR_2way_MIMO: {SKR_2way_MIMO}")

    return {
        "NT": NT, "distance_km": distance_km,
        "QBER_2way_MIMO": QBER_2way_MIMO, "SKR_2way_MIMO": SKR_2way_MIMO,
        "sifted_key_length": len(bob_key_bits), "key_match": key_match,
        "eve_stolen_both_phases": len(set(eve.stored_photons_phase1) & set(eve.stored_photons_phase2)),
    }

In [ ]:
# ================= Step 31: Loop Over MIMO_configs and distances_km_graph =================
# (Function combining Steps 7-30 is shown above in our discussion)

results_lm05 = []

for NT in MIMO_configs:
    for distance_km in distances_km_graph:
        result = run_lm05_mimo_fso_simulation(NT, distance_km, verbose=False)
        results_lm05.append(result)
        print(f"[Summary] NT={NT} distance={distance_km}km -> QBER={result['QBER_2way_MIMO']:.6f} "
              f"SKR={result['SKR_2way_MIMO']:.6f} sifted_len={result['sifted_key_length']} "
              f"key_match={result['key_match']}")

print()
print(f"Completed {len(results_lm05)} combinations ({len(MIMO_configs)} MIMO configs x {len(distances_km_graph)} distances).")

In [ ]:
# ================= Step 32: Generate Performance Graphs =================
import plotly.graph_objects as go

QBER_by_NT = {NT: [] for NT in MIMO_configs}
SKR_by_NT = {NT: [] for NT in MIMO_configs}

for r in results_lm05:
    QBER_by_NT[r["NT"]].append(r["QBER_2way_MIMO"])
    SKR_by_NT[r["NT"]].append(r["SKR_2way_MIMO"])

# ---- Plot: QBER vs Distance ----
fig_qber = go.Figure()
for NT in MIMO_configs:
    fig_qber.add_trace(go.Scatter(x=distances_km_graph, y=QBER_by_NT[NT],
                                   mode="lines+markers", name=f"NT=NR={NT}"))
fig_qber.update_layout(title="QBER vs Distance (LM05 Two-Way)",
                        xaxis_title="Distance (km)", yaxis_title="QBER_2way_MIMO",
                        template="plotly_white")
fig_qber.show()

# ---- Plot: SKR vs Distance ----
fig_skr = go.Figure()
for NT in MIMO_configs:
    fig_skr.add_trace(go.Scatter(x=distances_km_graph, y=SKR_by_NT[NT],
                                  mode="lines+markers", name=f"NT=NR={NT}"))
fig_skr.update_layout(title="SKR vs Distance (LM05 Two-Way)",
                       xaxis_title="Distance (km)", yaxis_title="SKR_2way_MIMO (bps)",
                       yaxis_type="log", template="plotly_white")
fig_skr.show()

In [ ]:
# ================= ADD THIS AS A NEW CELL AT THE END OF SeQUeNCe_LM05_Simulator_8.ipynb =================
# Saves the LM05 sweep results (the `results_lm05` list from Step 31) to a JSON file
# so the comparison notebook can load it.

import json

lm05_export = [
    {
        "NT": r["NT"],
        "distance_km": int(r["distance_km"]),
        "QBER": float(r["QBER_2way_MIMO"]),
        "SKR": float(r["SKR_2way_MIMO"]),
        "sifted_key_length": int(r["sifted_key_length"]),
        "key_match": bool(r["key_match"]),
    }
    for r in results_lm05
]

with open("lm05_results.json", "w") as f:
    json.dump(lm05_export, f, indent=2)

print(f"Saved {len(lm05_export)} LM05 results -> lm05_results.json")